In [ ]:
def _add_coal_bid_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Features derived from BIDDAYOFFER_D declared maximum availability for
    NSW coal generators — the closest available substitute for PDPASA
    scheduled-outage data.

    MAXAVAIL is submitted by generators before 12:30 the day prior.
    A low value signals a unit is booked off for scheduled maintenance.
    No-op if coal_declared_avail is absent or all-NaN.
    """
    df = df.copy()
    if "coal_declared_avail" not in df.columns or df["coal_declared_avail"].isna().all():
        return df

    cda = df["coal_declared_avail"]

    # Direct declared availability signal
    df["coal_declared_avail"] = cda.astype(np.float32)

    # Ratio: declared vs actual dispatched coal (deviation flags maintenance)
    if "coal_mw" in df.columns:
        df["coal_bid_vs_actual"]     = (cda - df["coal_mw"]).clip(-5000, 5000).astype(np.float32)
        df["coal_bid_util"]          = (df["coal_mw"] / (cda + 1)).clip(0, 1.5).astype(np.float32)

    # Rolling trend: is declared availability declining? (more units coming offline)
    df["coal_bid_rmean_48"]      = cda.rolling(48).mean().astype(np.float32)
    df["coal_bid_rmin_48"]       = cda.rolling(48).min().astype(np.float32)
    df["coal_bid_trend_48"]      = (cda - cda.shift(48)).astype(np.float32)   # 24h change
    df["coal_bid_trend_96"]      = (cda - cda.shift(96)).astype(np.float32)   # 48h change

    # Low declared availability flag (major maintenance period)
    df["coal_bid_low"]           = (cda < NSW_COAL_MAX_MW * 0.65).astype(np.float32)
    df["coal_bid_low_count_48"]  = df["coal_bid_low"].rolling(48).sum().astype(np.float32)

    # Lags so the model can see the recent bid history
    for lag in [1, 2, 48, 96, 336]:
        df[f"coal_bid_lag_{lag}"] = cda.shift(lag).astype(np.float32)

    return df